# Vectorized QMC (Bayesian)

Demonstrates Bayesian cubature rules (`CubQMCBayesLatticeG` and `CubQMCBayesNetG`) for integration, which model the integrand as a Gaussian process and provide posterior credible intervals as error estimates.

Original QMCPy demo: [`QMCPy/demos/vectorized_qmc_bayes.ipynb`](../../QMCPy/demos/vectorized_qmc_bayes.ipynb)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QMCSoftware/QMC.jl/blob/develop/demos/vectorized_qmc_bayes.ipynb)

*As in the non-Bayesian Julia counterpart, this notebook concentrates on the Bayesian vectorized cubature features that have direct `QMC.jl` support.*

In [1]:
using QMC
using LinearAlgebra: Diagonal
import QMC: Uniform
using Statistics
using Printf

const HAVE_PLOTS = Base.find_package("Plots") !== nothing
HAVE_PLOTS && @eval using Plots


## LD Sequence

Compare IID, digital net, and lattice point sets.

In [2]:
n = 2^6
for (dd, name) in [
    (IIDStdUniform(2; seed=7),  "IID"),
    (DigitalNetB2(2; seed=7),   "Digital Net"),
    (Lattice(2; seed=7),        "Lattice"),
]
    pts = gen_samples(dd, n)
    println("$name ($n points):")
    println("  Mean:  $(round.(mean(pts, dims=1), digits=3))")
    println("  Range: [$(round.(minimum(pts, dims=1), digits=3)), " *
            "$(round.(maximum(pts, dims=1), digits=3))]")
end

IID (64 points):


  Mean:  [0.506 0.422]


  Range: [[0.0 0.027], [0.988 0.977]]


Digital Net (64 points):


  Mean:  [0.497 0.503]
  Range: [[0.004 0.011], [0.989 0.995]]
Lattice (64 points):


  Mean:  [0.507 0.499]
  Range: [[0.015 0.007], [0.999 0.991]]


In [ ]:
if HAVE_PLOTS
    n_s = 2^6
    sampler_specs = [
        ("IID",         IIDStdUniform(2; seed=7),  :tomato),
        ("Digital Net", DigitalNetB2(2; seed=7),   :steelblue),
        ("Lattice",     Lattice(2; seed=7),         :green),
    ]
    scatter_plts = []
    for (name, dd, col) in sampler_specs
        pts = gen_samples(dd, n_s)
        subplot = scatter(pts[:, 1], pts[:, 2];
            markersize=4, alpha=0.8, color=col, label=nothing,
            xlabel="x₁", ylabel="x₂",
            title="$name (n=$n_s)",
            xlims=(0, 1), ylims=(0, 1), aspect_ratio=:equal)
        push!(scatter_plts, subplot)
    end
    display(plot(scatter_plts...; layout=(1, 3), size=(900, 320),
        plot_title="Low-discrepancy sequences"))
end


## Simple Example

3D input → 2D output: displacement D and stress S as functions of Young's modulus E, horizontal load X, and vertical load Y.

In [3]:
function cantilever_beam(x)
    l, w, t = 100.0, 4.0, 2.0
    E, X, Y = x[1], x[2], x[3]
    D = 4l^3 / (E * w * t) * sqrt(X^2/t^4 + Y^2/w^4)
    S = 600 * (X / (w * t^2) + Y / (w^2 * t))
    return D, S
end

# Evaluate over points using the same Gaussian input model as QMCPy
dd = DigitalNetB2(3; seed=7, graycode=false)
tm_beam = Gaussian(dd;
    mean=[2.9e7, 500.0, 1000.0],
    covariance=Diagonal([(1.45e6)^2, (100.0)^2, (100.0)^2]))
u = gen_samples(dd, 1024)
x = transform(tm_beam, u)
D_vals = Float64[]
S_vals = Float64[]
for i in 1:size(x, 1)
    disp, stress = cantilever_beam(x[i, :])
    push!(D_vals, disp)
    push!(S_vals, stress)
end

println("Displacement D: mean=$(round(mean(D_vals), digits=4)), std=$(round(std(D_vals), digits=4))")
println("Stress S:       mean=$(round(mean(S_vals), digits=4)), std=$(round(std(S_vals), digits=4))")

Displacement D: mean=2.425, std=0.4047


Stress S:       mean=37489.9412, std=4194.3089


## Bayesian QMC Integration

Use the Bayesian stopping criteria which model the integrand as a sample path from a Gaussian process to obtain credible-interval-based error bounds.

In [4]:
# --- Bayesian Lattice ---
println("=== CubQMCBayesLatticeG ===")
let
    for dim in [1, 3, 5]
        dd_lattice = Lattice(dim; seed=7)
        tm_lattice = Gaussian(dd_lattice; mean=0.0, covariance=0.5)
        f_lattice = Keister(tm_lattice)
        sc_lattice = CubQMCBayesLatticeG(f_lattice; abs_tol=1e-3)
        result_lattice = integrate(sc_lattice)
        exact_lattice = keister_exact(dim)
        err = abs(result_lattice.solution - exact_lattice)
        println("  d=$dim: sol=$(round(result_lattice.solution, digits=6)), " *
                "err=$(round(err, sigdigits=2)), " *
                "n=$(result_lattice.data[:n])")
    end
end

=== CubQMCBayesLatticeG ===
  d=1: sol=1.380388, err=2.4e-8, n=256


  d=3: sol=2.167373, err=0.00094, n=256
  d=5: sol=0.997528, err=0.14, n=256


In [5]:
# --- Bayesian Digital Net ---
println("=== CubQMCBayesNetG ===")
let
    for dim in [1, 3, 5]
        dd_net = DigitalNetB2(dim; seed=7)
        tm_net = Gaussian(dd_net; mean=0.0, covariance=0.5)
        f_net = Keister(tm_net)
        sc_net = CubQMCBayesNetG(f_net; abs_tol=1e-3)
        result_net = integrate(sc_net)
        exact_net = keister_exact(dim)
        err = abs(result_net.solution - exact_net)
        println("  d=$dim: sol=$(round(result_net.solution, digits=6)), " *
                "err=$(round(err, sigdigits=2)), " *
                "n=$(result_net.data[:n])")
    end
end

=== CubQMCBayesNetG ===
  d=1: sol=1.380452, err=6.3e-5, n=256


  d=3: sol=2.170933, err=0.0026, n=512
  d=5: sol=1.157681, err=0.022, n=8192


## Comparison: Bayesian vs Frequentist QMC

Compare the Bayesian stopping criteria with their frequentist counterparts (`CubQMCLatticeG`, `CubQMCNetG`) for the same tolerance.

As in the non-Bayesian vectorized Julia demo, the broader QMCPy notebook also shows BO QEI, Bayesian logistic regression, Ishigami, and neural-network examples. This Julia notebook keeps the directly supported Bayesian cubature pieces and leaves those additional application sections to package-specific Julia workflows.

In [6]:
d = 3
tol = 1e-3
exact = keister_exact(d)

println("Keister integral (d=$d), abs_tol=$tol\n")
println("Method                      Solution     Error       n")
println("-"^60)

let
    for (name, make_sc) in [
        ("CubQMCLatticeG",      () -> begin
            dd_cmp = Lattice(d; seed=7)
            tm_cmp = Gaussian(dd_cmp; mean=0.0, covariance=0.5)
            CubQMCLatticeG(Keister(tm_cmp); abs_tol=tol)
        end),
        ("CubQMCBayesLatticeG", () -> begin
            dd_cmp = Lattice(d; seed=7)
            tm_cmp = Gaussian(dd_cmp; mean=0.0, covariance=0.5)
            CubQMCBayesLatticeG(Keister(tm_cmp); abs_tol=tol)
        end),
        ("CubQMCNetG",          () -> begin
            dd_cmp = DigitalNetB2(d; seed=7, randomize="LMS_DS", graycode=false)
            tm_cmp = Gaussian(dd_cmp; mean=0.0, covariance=0.5)
            CubQMCNetG(Keister(tm_cmp); abs_tol=tol)
        end),
        ("CubQMCBayesNetG",     () -> begin
            dd_cmp = DigitalNetB2(d; seed=7)
            tm_cmp = Gaussian(dd_cmp; mean=0.0, covariance=0.5)
            CubQMCBayesNetG(Keister(tm_cmp); abs_tol=tol)
        end),
    ]
        sc_cmp = make_sc()
        result_cmp = integrate(sc_cmp)
        err = abs(result_cmp.solution - exact)
        @printf("%-28s %.6f    %.2e    %d\n", name, result_cmp.solution, err, result_cmp.data[:n])
    end
end

Keister integral (d=3), abs_tol=0.001

Method                      Solution     Error       n
------------------------------------------------------------
CubQMCLatticeG               2.168409    1.00e-04    2048
CubQMCBayesLatticeG          2.167373    9.36e-04    256


CubQMCNetG                   2.168554    2.45e-04    16384
CubQMCBayesNetG              2.170933    2.62e-03    512


## Convergence Comparison


In [ ]:
if HAVE_PLOTS
    # Keister convergence: Bayesian vs frequentist, lattice vs digital net
    d_c, tol_c = 3, 1e-3
    exact_c = keister_exact(d_c)
    ns_c = [2^k for k in 5:12]

    function keister_err(make_dd)
        [abs(mean(evaluate(Keister(Gaussian(make_dd(seed),
            mean=0.0, covariance=0.5)), gen_samples(make_dd(seed), n))) - exact_c)
         for (seed, n) in zip(1:length(ns_c), ns_c)]
    end

    err_lat  = keister_err(s -> Lattice(d_c; seed=s))
    err_net  = keister_err(s -> DigitalNetB2(d_c; seed=s, graycode=false))
    err_iid  = keister_err(s -> IIDStdUniform(d_c; seed=s))

    p = plot(ns_c, err_lat;
        xscale=:log2, yscale=:log10,
        marker=:circle, lw=2, ms=5, color=:steelblue,
        label="Lattice",
        xlabel="n", ylabel="|error|",
        title="Keister convergence (d=$d_c)")
    plot!(ns_c, err_net; marker=:diamond, lw=2, ms=5, color=:green, label="Digital Net B2")
    plot!(ns_c, err_iid; marker=:square, lw=2, ms=5, color=:tomato, linestyle=:dash, label="IID")
    ref = Float64.(ns_c)
    plot!(ref, 0.3 .* ref .^ (-0.5); lw=1, color=:black, linestyle=:dot, label="n^{-0.5}")
    display(p)
end
